# Olist E-Commerce BI — Week 2: KPIs, Retention & Customer Segments

**Author:** Oluwaseyi ([@shykeys-data](https://github.com/shykeys-data)) · **Builds on:** [Week 1: data pipeline & star schema](01_data_pipeline_star_schema.ipynb)
**Tools:** Python · pandas · DuckDB (SQL) · Plotly · SciPy · Google Colab

---

### The question this week
Week 1 found that **only 3.1% of customers ever placed a second order**. This notebook asks why, and what it means for the business:

| Section | Business question | Method |
|---|---|---|
| 1. KPI layer | How is the business doing month by month? | SQL on the star schema |
| 2. Cohort retention | Do customers come back, and when? | Monthly cohort matrix |
| 3. RFM segmentation | Which customers matter most, and who is slipping away? | Recency, Frequency, Monetary scoring |
| 4. Delivery vs reviews | Do late deliveries hurt ratings and repeat buying? | Comparison + significance tests |
| 5. Where the money is | Which regions and categories drive revenue? | SQL breakdowns |
| 6. Key findings | What should Olist do? | Summary |

> **How to run:** same as Week 1. Add `KAGGLE_API_TOKEN` to Colab **Secrets** (🔑), then **Runtime → Run all**. Section 0 rebuilds the Week 1 model automatically, so this notebook runs on its own.

## 0. Setup and rebuild the Week 1 model

In [ ]:
!pip install -q -U duckdb kagglehub kaggle

import os, glob
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
BLUE = "#2E6FDB"

In [ ]:
# Download the data (same as Week 1: keys come from Colab Secrets)
DATA_DIR = os.environ.get("OLIST_DATA_DIR")
if DATA_DIR is None:
    from google.colab import userdata
    def get_secret(name):
        try:
            return userdata.get(name)
        except Exception:
            return None
    if get_secret("KAGGLE_API_TOKEN"):
        os.environ["KAGGLE_API_TOKEN"] = get_secret("KAGGLE_API_TOKEN")
    if get_secret("KAGGLE_USERNAME") and get_secret("KAGGLE_KEY"):
        os.environ["KAGGLE_USERNAME"] = get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = get_secret("KAGGLE_KEY")
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("Data folder:", DATA_DIR)

This cell rebuilds the Week 1 star schema with **the same cleaning rules**, condensed into one step. Colab deletes files when a session ends, so rebuilding is more reliable than loading last week's saved files.

In [ ]:
def load(name):
    path = glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True)[0]
    return pd.read_csv(path, encoding="utf-8-sig")

raw = {k: load(f) for k, f in {
    "orders": "olist_orders_dataset.csv", "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv", "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv", "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv", "translation": "product_category_name_translation.csv",
}.items()}

# Orders: dates and delivery metrics
orders = raw["orders"].copy()
for c in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
          "order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")
orders.loc[orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"], "order_delivered_customer_date"] = pd.NaT
delivered_mask = orders["order_delivered_customer_date"].notna()
orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.total_seconds() / 86400
orders["days_late"] = (orders["order_delivered_customer_date"].dt.normalize() - orders["order_estimated_delivery_date"].dt.normalize()).dt.days
orders["is_late"] = (orders["days_late"] > 0).astype("boolean").where(delivered_mask)

# Items, payments, reviews: one row per order where needed
items = raw["items"].copy()
items["item_total"] = items["price"] + items["freight_value"]
items_by_order = items.groupby("order_id").agg(n_items=("order_item_id", "count"),
                                               items_value=("price", "sum"),
                                               freight_value=("freight_value", "sum")).reset_index()
pay = raw["payments"]
pay = pay[~((pay["payment_type"] == "not_defined") & (pay["payment_value"] == 0))]
main_type = pay.sort_values("payment_value", ascending=False).drop_duplicates("order_id")[["order_id", "payment_type"]]
payments_by_order = (pay.groupby("order_id").agg(total_paid=("payment_value", "sum"),
                                                  installments=("payment_installments", "max"))
                     .reset_index().merge(main_type.rename(columns={"payment_type": "main_payment_type"}), on="order_id"))
reviews = raw["reviews"].copy()
for c in ["review_creation_date", "review_answer_timestamp"]:
    reviews[c] = pd.to_datetime(reviews[c], errors="coerce")
reviews_by_order = reviews.sort_values("review_answer_timestamp").drop_duplicates("order_id", keep="last")[["order_id", "review_score", "review_creation_date"]]

# Products with English categories
products = raw["products"].copy()
translation = dict(zip(raw["translation"]["product_category_name"], raw["translation"]["product_category_name_english"]))
translation.setdefault("pc_gamer", "pc_gamer")
translation.setdefault("portateis_cozinha_e_preparadores_de_alimentos", "portable_kitchen_and_food_preparers")
products["product_category_name"] = products["product_category_name"].fillna("unknown")
products["category"] = (products["product_category_name"].map(translation).fillna(products["product_category_name"])
                        .str.replace("_", " ").str.title())
dim_product = products[["product_id", "category"]]

# Facts
fact_orders = (orders.merge(raw["customers"][["customer_id", "customer_unique_id"]], on="customer_id", how="left")
               .merge(items_by_order, on="order_id", how="left")
               .merge(payments_by_order, on="order_id", how="left")
               .merge(reviews_by_order, on="order_id", how="left"))
fact_orders["n_items"] = fact_orders["n_items"].fillna(0).astype(int)
fact_orders["order_value"] = fact_orders["items_value"] + fact_orders["freight_value"]
fact_orders["purchase_date_key"] = fact_orders["order_purchase_timestamp"].dt.strftime("%Y%m%d").astype(int)
fact_order_items = items.merge(fact_orders[["order_id", "customer_unique_id", "purchase_date_key", "order_status"]], on="order_id", how="left")

# Dimensions
cust = raw["customers"].merge(orders[["customer_id", "order_purchase_timestamp"]], on="customer_id", how="left")
dim_customer = (cust.sort_values("order_purchase_timestamp").drop_duplicates("customer_unique_id", keep="last")
                [["customer_unique_id", "customer_city", "customer_state"]]
                .rename(columns={"customer_city": "city", "customer_state": "state"}))
dates = pd.date_range(orders["order_purchase_timestamp"].min().normalize(), orders["order_estimated_delivery_date"].max().normalize())
dim_date = pd.DataFrame({"date_key": dates.strftime("%Y%m%d").astype(int), "date": dates,
                         "year": dates.year, "month": dates.month, "year_month": dates.strftime("%Y-%m")})

# Quick integrity check: same totals as Week 1
assert len(fact_orders) == len(raw["orders"]) and len(fact_order_items) == len(raw["items"])
assert np.isclose(fact_order_items["item_total"].sum(), fact_orders["order_value"].sum())
print(f"Model rebuilt ✅  {len(fact_orders):,} orders · {len(fact_order_items):,} items · {len(dim_customer):,} customers")

### Defining a "purchase"
For customer behaviour we count **every order except canceled or unavailable ones**: those were never real purchases. Each customer's **first purchase month** is their **cohort**.

In [ ]:
purchases = fact_orders[~fact_orders["order_status"].isin(["canceled", "unavailable"])].copy()
purchases["order_value"] = purchases["order_value"].fillna(0)
purchases["order_month"] = purchases["order_purchase_timestamp"].dt.to_period("M")

first = purchases.groupby("customer_unique_id")["order_purchase_timestamp"].min().rename("first_purchase")
purchases = purchases.join(first, on="customer_unique_id")
purchases["cohort"] = purchases["first_purchase"].dt.to_period("M")

dim_customer = dim_customer.merge(first, on="customer_unique_id", how="left")
dim_customer["first_purchase_month"] = dim_customer["first_purchase"].dt.strftime("%Y-%m")

orders_per_customer = purchases.groupby("customer_unique_id")["order_id"].nunique()
print(f"Purchases: {len(purchases):,} by {orders_per_customer.size:,} customers")
print(f"Customers with 2+ purchases: {(orders_per_customer > 1).mean():.1%}")
print(orders_per_customer.value_counts().sort_index().rename("customers").to_frame().rename_axis("purchases").head(6))

## 1. KPI layer: the business month by month
One SQL query turns the star schema into a monthly KPI table: the kind of table that sits behind a management dashboard. Each KPI has a written definition, so nobody argues about what a number means.

| KPI | Definition |
|---|---|
| Revenue | Price + freight of delivered orders |
| AOV | Revenue ÷ delivered orders |
| New-customer share | % of orders placed in the customer's first purchase month |
| Late rate | % of delivered orders arriving after the promised date |
| Avg review | Mean review score (1–5) |

In [ ]:
import duckdb

kpis = duckdb.sql('''
    SELECT d.year_month,
           COUNT(*)                                       AS orders,
           ROUND(SUM(f.order_value), 0)                   AS revenue,
           ROUND(AVG(f.order_value), 2)                   AS aov,
           COUNT(DISTINCT f.customer_unique_id)           AS customers,
           ROUND(100.0 * AVG(CASE WHEN c.first_purchase_month = d.year_month THEN 1 ELSE 0 END), 1)
                                                          AS new_customer_share_pct,
           ROUND(100.0 * AVG(CAST(f.is_late AS DOUBLE)), 1) AS late_rate_pct,
           ROUND(AVG(f.review_score), 2)                  AS avg_review
    FROM fact_orders f
    JOIN dim_date d     ON f.purchase_date_key = d.date_key
    JOIN dim_customer c ON f.customer_unique_id = c.customer_unique_id
    WHERE f.order_status = 'delivered'
    GROUP BY d.year_month
    ORDER BY d.year_month
''').df()

kpis = kpis[(kpis["year_month"] >= "2017-01") & (kpis["year_month"] <= "2018-08")].reset_index(drop=True)
kpis

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Revenue (R$)", "Late deliveries (%)", "Average review score (1–5)"))
fig.add_bar(x=kpis["year_month"], y=kpis["revenue"], marker_color=BLUE, row=1, col=1)
fig.add_scatter(x=kpis["year_month"], y=kpis["late_rate_pct"], mode="lines+markers", line=dict(color=BLUE, width=2), row=2, col=1)
fig.add_scatter(x=kpis["year_month"], y=kpis["avg_review"], mode="lines+markers", line=dict(color=BLUE, width=2), row=3, col=1)
fig.update_layout(template="simple_white", height=650, showlegend=False,
                  title="Monthly KPIs, Jan 2017 – Aug 2018")
fig.show()

**What to look for:** months where the late rate jumps (for example after Black Friday, and in early 2018) should line up with dips in the review score. Section 4 tests that link properly.

## 2. Cohort retention: do customers come back?
Customers are grouped by the month of their first purchase (their **cohort**). For each cohort we track what % placed another order 1, 2, 3… months later.

In [ ]:
def month_index(p):            # Period → running month number
    return p.dt.year * 12 + p.dt.month

purchases["months_since_first"] = month_index(purchases["order_month"]) - month_index(purchases["cohort"])

cohort_counts = (purchases.groupby(["cohort", "months_since_first"])["customer_unique_id"].nunique()
                 .unstack(fill_value=0))
cohort_size = cohort_counts[0]
retention = cohort_counts.div(cohort_size, axis=0) * 100

# Keep complete cohorts and the first 12 months
retention = retention.loc[(retention.index >= pd.Period("2017-01", "M")) & (retention.index <= pd.Period("2018-06", "M"))]
retention = retention.reindex(columns=range(0, 13))
retention.index = retention.index.astype(str)
retention.insert(0, "customers", cohort_size.reindex(pd.PeriodIndex(retention.index, freq="M")).values)
retention.round(2)

In [ ]:
heat = retention.drop(columns=["customers", 0])
fig = px.imshow(heat, color_continuous_scale="Blues", aspect="auto", text_auto=".1f",
                labels=dict(x="Months after first purchase", y="Cohort (first purchase month)", color="% returned"),
                title="Cohort retention: % of customers who ordered again, by month")
fig.update_layout(template="simple_white", height=600)
fig.show()

In [ ]:
# How long until a second order, and how many customers ever get there?
order_seq = (purchases.sort_values("order_purchase_timestamp")
             .drop_duplicates(["customer_unique_id", "order_id"])
             .assign(n=lambda d: d.groupby("customer_unique_id").cumcount() + 1))
second = order_seq[order_seq["n"] == 2].set_index("customer_unique_id")["order_purchase_timestamp"]
gap_days = (second - first.reindex(second.index)).dt.days

end_of_data = purchases["order_purchase_timestamp"].max()
rows = []
for window in [30, 90, 180, 365]:
    eligible = first[first <= end_of_data - pd.Timedelta(days=window)]      # customers we could observe for the full window
    came_back = gap_days.reindex(eligible.index).le(window).sum()
    rows.append({"window": f"within {window} days", "eligible_customers": len(eligible),
                 "came_back": int(came_back), "repeat_rate_%": round(100 * came_back / len(eligible), 2)})
repeat_windows = pd.DataFrame(rows)

print(f"Median days to a 2nd order (repeat buyers only): {gap_days.median():.0f}")
print(f"Repeat buyers who came back within 30 days: {(gap_days <= 30).mean():.0%}")
repeat_windows

**Reading the cohorts:** if most cells after month 0 are below 1%, Olist is essentially a **one-time-purchase business**: growth comes almost entirely from new customers. The repeat-window table is the fairer comparison, because it only counts customers we could actually watch for the full window.

## 3. RFM segmentation: who are the customers?
Each customer gets three scores:
- **Recency (R):** days since their last purchase. Recent = better (score 5).
- **Frequency (F):** number of purchases. With ~97% one-time buyers, quintiles don't work, so F is **1, 2 or 3+**.
- **Monetary (M):** total spent, scored 1–5 by quintile.

The scores are then turned into **plain-English segments** a marketing team can act on.

In [ ]:
snapshot = end_of_data + pd.Timedelta(days=1)
rfm = (purchases.groupby("customer_unique_id")
       .agg(last_purchase=("order_purchase_timestamp", "max"),
            frequency=("order_id", "nunique"),
            monetary=("order_value", "sum")))
rfm["recency_days"] = (snapshot - rfm["last_purchase"]).dt.days

rfm["R"] = pd.qcut(rfm["recency_days"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["F"] = rfm["frequency"].clip(upper=3)

conditions = [
    (rfm["F"] >= 2) & (rfm["R"] >= 4),
    (rfm["F"] >= 2),
    (rfm["F"] == 1) & (rfm["R"] >= 4) & (rfm["M"] >= 4),
    (rfm["F"] == 1) & (rfm["R"] >= 4),
    (rfm["F"] == 1) & (rfm["R"] <= 2) & (rfm["M"] >= 4),
    (rfm["F"] == 1) & (rfm["R"] <= 2),
]
segments = ["Champions", "Loyal (lapsing)", "New big spenders", "New customers",
            "Big spenders at risk", "Hibernating"]
rfm["segment"] = np.select(conditions, segments, default="Needs attention")

segment_summary = (rfm.groupby("segment")
                   .agg(customers=("monetary", "size"), revenue=("monetary", "sum"),
                        avg_spend=("monetary", "mean"), avg_recency_days=("recency_days", "mean"),
                        avg_purchases=("frequency", "mean")))
segment_summary["customers_%"] = 100 * segment_summary["customers"] / segment_summary["customers"].sum()
segment_summary["revenue_%"] = 100 * segment_summary["revenue"] / segment_summary["revenue"].sum()
segment_summary = segment_summary.sort_values("revenue", ascending=False)
segment_summary.round(1)

| Segment | Rule | Suggested action |
|---|---|---|
| Champions | 2+ purchases, bought recently | VIP perks, referral programme |
| Loyal (lapsing) | 2+ purchases, not recently | Personal win-back offer |
| New big spenders | 1 high-value purchase, recent | Fast second-purchase nudge, cross-sell |
| New customers | 1 purchase, recent | Onboarding emails, first-repeat discount |
| Needs attention | 1 purchase, mid recency | Reminder with recommendations |
| Big spenders at risk | 1 high-value purchase, long ago | Targeted win-back on their category |
| Hibernating | 1 low-value purchase, long ago | Low-cost reactivation only |

In [ ]:
seg_plot = segment_summary.reset_index().melt(id_vars="segment", value_vars=["customers_%", "revenue_%"],
                                              var_name="measure", value_name="percent")
seg_plot["measure"] = seg_plot["measure"].map({"customers_%": "% of customers", "revenue_%": "% of revenue"})
fig = px.bar(seg_plot, x="percent", y="segment", color="measure", barmode="group", orientation="h",
             color_discrete_map={"% of customers": "#A9C4F5", "% of revenue": BLUE},
             title="RFM segments: share of customers vs share of revenue",
             labels={"percent": "%", "segment": "", "measure": ""})
fig.update_layout(template="simple_white", height=480, yaxis={"categoryorder": "total ascending"})
fig.show()

## 4. Do late deliveries hurt reviews and repeat purchases?
A hypothesis is only useful if it's tested. Two questions:
1. Do late orders get **lower review scores**?
2. Are customers whose **first** order arrived late **less likely to come back**?

In [ ]:
rated = fact_orders[(fact_orders["order_status"] == "delivered") & fact_orders["review_score"].notna()
                    & fact_orders["is_late"].notna()].copy()
rated["delivery"] = np.where(rated["is_late"].astype(bool), "Late", "On time or early")

late_vs_review = (rated.groupby("delivery")
                  .agg(orders=("order_id", "count"), avg_review=("review_score", "mean"),
                       one_star_pct=("review_score", lambda s: 100 * (s == 1).mean()),
                       five_star_pct=("review_score", lambda s: 100 * (s == 5).mean())))
late_vs_review.round(2)

In [ ]:
# Is the difference real or just chance? Mann-Whitney U test (review scores are ordinal, not normal)
late_scores = rated.loc[rated["delivery"] == "Late", "review_score"]
ontime_scores = rated.loc[rated["delivery"] == "On time or early", "review_score"]
u_stat, p_value = stats.mannwhitneyu(ontime_scores, late_scores, alternative="greater")
print(f"Average review, on time: {ontime_scores.mean():.2f}  vs  late: {late_scores.mean():.2f}  "
      f"(difference {ontime_scores.mean() - late_scores.mean():.2f} stars)")
print(f"Mann-Whitney U p-value: {p_value:.2e}  →  {'statistically significant' if p_value < 0.05 else 'not significant'} at 5%")

In [ ]:
# The more late, the worse? Review score by how early/late the order arrived
bins = [-np.inf, -15, -8, -1, 0, 3, 7, 14, np.inf]
labels = ["15+ days early", "8–14 days early", "1–7 days early", "On the promised day",
          "1–3 days late", "4–7 days late", "8–14 days late", "15+ days late"]
rated["timing"] = pd.cut(rated["days_late"], bins=bins, labels=labels)
by_timing = (rated.groupby("timing", observed=False)
             .agg(orders=("order_id", "count"), avg_review=("review_score", "mean"),
                  one_star_pct=("review_score", lambda s: 100 * (s == 1).mean()))
             .reset_index())
by_timing.round(2)

In [ ]:
fig = px.bar(by_timing, x="timing", y="avg_review", text=by_timing["avg_review"].round(2),
             title="Average review score by delivery timing",
             labels={"timing": "", "avg_review": "Average review (1–5)"},
             hover_data={"orders": ":,", "one_star_pct": ":.1f"})
fig.update_traces(marker_color=BLUE, textposition="outside")
fig.update_layout(template="simple_white", height=450, yaxis_range=[0, 5.4])
fig.show()

In [ ]:
# Does a late FIRST order make customers less likely to return?
first_orders = (purchases.sort_values("order_purchase_timestamp")
                .drop_duplicates("customer_unique_id")[["customer_unique_id", "is_late"]]
                .dropna(subset=["is_late"]))
first_orders["came_back"] = first_orders["customer_unique_id"].map(orders_per_customer) > 1
first_orders["first_order"] = np.where(first_orders["is_late"].astype(bool), "Late", "On time or early")

repeat_by_first = (first_orders.groupby("first_order")
                   .agg(customers=("came_back", "size"), repeat_rate_pct=("came_back", lambda s: 100 * s.mean())))
chi2, p_repeat, _, _ = stats.chi2_contingency(pd.crosstab(first_orders["first_order"], first_orders["came_back"]))
print(f"Chi-square p-value: {p_repeat:.3g}  →  {'significant' if p_repeat < 0.05 else 'not significant'} at 5%")
repeat_by_first.round(2)

## 5. Where the money is: regions and categories

In [ ]:
by_state = duckdb.sql('''
    SELECT c.state,
           COUNT(*)                                          AS orders,
           ROUND(SUM(f.order_value), 0)                      AS revenue,
           ROUND(100.0 * SUM(f.order_value) / SUM(SUM(f.order_value)) OVER (), 1) AS revenue_share_pct,
           ROUND(AVG(f.delivery_days), 1)                    AS avg_delivery_days,
           ROUND(100.0 * AVG(CAST(f.is_late AS DOUBLE)), 1)  AS late_rate_pct,
           ROUND(AVG(f.review_score), 2)                     AS avg_review
    FROM fact_orders f
    JOIN dim_customer c ON f.customer_unique_id = c.customer_unique_id
    WHERE f.order_status = 'delivered'
    GROUP BY c.state
    ORDER BY revenue DESC
''').df()
by_state.head(10)

In [ ]:
by_category = duckdb.sql('''
    SELECT p.category,
           COUNT(DISTINCT i.order_id)                        AS orders,
           ROUND(SUM(i.item_total), 0)                       AS revenue,
           ROUND(100.0 * SUM(i.item_total) / SUM(SUM(i.item_total)) OVER (), 1) AS revenue_share_pct,
           ROUND(AVG(i.price), 2)                            AS avg_item_price
    FROM fact_order_items i
    JOIN dim_product p ON i.product_id = p.product_id
    WHERE i.order_status = 'delivered'
    GROUP BY p.category
    ORDER BY revenue DESC
''').df()
by_category.head(10)

In [ ]:
top_cat = by_category.head(10).sort_values("revenue")
fig = px.bar(top_cat, x="revenue", y="category", orientation="h", text=top_cat["revenue_share_pct"].astype(str) + "%",
             title="Top 10 categories by revenue (delivered orders)", labels={"revenue": "Revenue (R$)", "category": ""})
fig.update_traces(marker_color=BLUE, textposition="outside")
fig.update_layout(template="simple_white", height=450, xaxis_tickformat=",.0f")
fig.show()

fig = px.scatter(by_state, x="avg_delivery_days", y="avg_review", size="orders", text="state",
                 title="Delivery time vs review score by state (bubble size = orders)",
                 labels={"avg_delivery_days": "Average delivery time (days)", "avg_review": "Average review (1–5)"})
fig.update_traces(marker_color=BLUE, textposition="top center")
fig.update_layout(template="simple_white", height=500)
fig.show()

## 6. Key findings (calculated live from the data)

In [ ]:
champ = segment_summary.loc["Champions"] if "Champions" in segment_summary.index else None
top3_states = by_state.head(3)
r_state = stats.spearmanr(by_state["avg_delivery_days"], by_state["avg_review"]).correlation

findings = [
    f"Retention: only {(orders_per_customer > 1).mean():.1%} of customers made a 2nd purchase; "
    f"{repeat_windows.iloc[1]['repeat_rate_%']:.1f}% did so within 90 days. Growth depends almost entirely on new customers.",
    f"Repeat buyers take a median of {gap_days.median():.0f} days to place a 2nd order; {(gap_days <= 30).mean():.0%} return within 30 days.",
    f"Delivery drives satisfaction: late orders average {late_scores.mean():.2f}★ vs {ontime_scores.mean():.2f}★ on time "
    f"(p = {p_value:.1e}); 1-star share rises from {late_vs_review.loc['On time or early', 'one_star_pct']:.0f}% "
    f"to {late_vs_review.loc['Late', 'one_star_pct']:.0f}%.",
    f"Customers whose first order arrived late came back {repeat_by_first.loc['Late', 'repeat_rate_pct']:.2f}% of the time vs "
    f"{repeat_by_first.loc['On time or early', 'repeat_rate_pct']:.2f}% when on time "
    f"(p = {p_repeat:.2g}, {'significant' if p_repeat < 0.05 else 'not significant'}).",
    f"Revenue is concentrated: {', '.join(top3_states['state'])} make up {top3_states['revenue_share_pct'].sum():.0f}% of revenue; "
    f"correlation between a state's delivery time and its average review: Spearman r = {r_state:.2f}.",
    f"Top category: {by_category.iloc[0]['category']} ({by_category.iloc[0]['revenue_share_pct']:.1f}% of revenue); "
    f"the top 10 categories make up {by_category.head(10)['revenue_share_pct'].sum():.0f}%.",
]
if champ is not None:
    findings.append(f"Champions are {champ['customers_%']:.1f}% of customers but {champ['revenue_%']:.1f}% of revenue.")

for i, f in enumerate(findings, 1):
    print(f"{i}. {f}\n")

## 7. Quality check: do the findings survive a closer look?
Good analysis means trying to break your own results before anyone else does. Each check below targets a specific way the findings above could be misleading.

| # | Risk | Check |
|---|---|---|
| QC1 | Some "repeat" orders are one shopping trip split into several orders on the same day | Count repeat buying by **distinct purchase days** instead of orders |
| QC2 | Customers who joined late in the data had less time to come back | Compare everyone over the **same 180-day window** |
| QC3 | Late orders cluster in certain months, which could fake a link to repeat buying | Compare late vs on-time **within the same cohort quarter** (Mantel-Haenszel test) |
| QC4 | Olist asks for a review around the promised date, so some late orders were rated before they arrived | Re-test using only reviews written **after delivery** |
| QC5 | A tiny p-value doesn't say how big an effect is | Report an **effect size** alongside the p-value |
| QC6 | States with very few orders can distort the state-level correlation | Re-run the correlation on states with **300+ orders** |
| QC7 | Numbers should add up across sections | **Reconciliation** checks |

In [ ]:
def fmt_p(p):
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

# QC1: purchase DAYS instead of orders
purchases["purchase_day"] = purchases["order_purchase_timestamp"].dt.normalize()
days_per_customer = purchases.groupby("customer_unique_id")["purchase_day"].nunique()
repeat_customers = orders_per_customer[orders_per_customer > 1].index
same_day_only = (days_per_customer.reindex(repeat_customers) == 1).sum()

day_seq = (purchases.drop_duplicates(["customer_unique_id", "purchase_day"])
           .sort_values("purchase_day")
           .assign(n=lambda d: d.groupby("customer_unique_id").cumcount() + 1))
second_day = day_seq[day_seq["n"] == 2].set_index("customer_unique_id")["purchase_day"]
gap_days_qc = (second_day - first.dt.normalize().reindex(second_day.index)).dt.days

qc1 = pd.DataFrame({
    "measure": ["Customers with 2+ purchases", "Median days to 2nd purchase", "Repeat buyers returning within 30 days"],
    "counting orders (original)": [f"{(orders_per_customer > 1).mean():.2%}", f"{gap_days.median():.0f}", f"{(gap_days <= 30).mean():.0%}"],
    "counting purchase days (QC)": [f"{(days_per_customer > 1).mean():.2%}", f"{gap_days_qc.median():.0f}", f"{(gap_days_qc <= 30).mean():.0%}"],
})
print(f"Repeat customers whose 'repeat' was really the same day: {same_day_only:,} of {len(repeat_customers):,} "
      f"({same_day_only / len(repeat_customers):.0%})")
qc1

In [ ]:
# QC2 + QC3: late first order vs coming back, on a fair and like-for-like basis
WINDOW = 180
fo = (purchases.sort_values("order_purchase_timestamp")
      .drop_duplicates("customer_unique_id")[["customer_unique_id", "order_purchase_timestamp", "is_late", "order_value"]]
      .dropna(subset=["is_late"]))
fo = fo[fo["order_purchase_timestamp"] <= end_of_data - pd.Timedelta(days=WINDOW)].copy()   # everyone observed for 180 days
fo["late"] = fo["is_late"].astype(bool)
fo["came_back"] = gap_days_qc.reindex(fo["customer_unique_id"]).le(WINDOW).fillna(False).astype(bool).values
fo["cohort_q"] = fo["order_purchase_timestamp"].dt.to_period("Q").astype(str)

qc2 = fo.groupby(np.where(fo["late"], "Late first order", "On-time first order")).agg(
    customers=("came_back", "size"), returned_within_180d_pct=("came_back", lambda s: 100 * s.mean()))
chi2_qc, p_qc2, _, _ = stats.chi2_contingency(pd.crosstab(fo["late"], fo["came_back"]))

# Mantel-Haenszel: pooled odds ratio across cohort quarters (compares like with like)
num = den = 0.0; a_sum = e_sum = v_sum = 0.0
for _, g in fo.groupby("cohort_q"):
    a = ((g["late"]) & (g["came_back"])).sum();  b = ((g["late"]) & (~g["came_back"])).sum()
    c = ((~g["late"]) & (g["came_back"])).sum(); d = ((~g["late"]) & (~g["came_back"])).sum()
    n = a + b + c + d
    if n < 2 or (a + b) == 0 or (c + d) == 0:
        continue
    num += a * d / n; den += b * c / n
    r1, c1 = a + b, a + c
    a_sum += a; e_sum += r1 * c1 / n
    v_sum += r1 * (c + d) * c1 * (b + d) / (n * n * (n - 1))
or_mh = num / den if den else np.nan
cmh_stat = (abs(a_sum - e_sum) - 0.5) ** 2 / v_sum if v_sum else np.nan
p_cmh = stats.chi2.sf(cmh_stat, 1) if v_sum else np.nan

print(f"Same 180-day window:  {fmt_p(p_qc2)} (chi-square)")
print(f"Within cohort quarters: odds ratio = {or_mh:.2f} (below 1 means late first orders return less), {fmt_p(p_cmh)} (Mantel-Haenszel)")
qc2.round(2)

In [ ]:
# QC4 + QC5: reviews written after delivery only, plus effect size
rated["review_after_delivery"] = rated["review_creation_date"] >= rated["order_delivered_customer_date"].dt.normalize()
before_share = (rated.groupby("delivery")["review_after_delivery"].apply(lambda s: 100 * (~s).mean()))

after = rated[rated["review_after_delivery"]]
late_a = after.loc[after["delivery"] == "Late", "review_score"]
ontime_a = after.loc[after["delivery"] == "On time or early", "review_score"]
u_a, p_a = stats.mannwhitneyu(ontime_a, late_a, alternative="greater")

def rank_biserial(u, n1, n2):          # 0 = no effect, 1 = every on-time review beats every late one
    return 2 * u / (n1 * n2) - 1

qc4 = pd.DataFrame({
    "test": ["All reviews (original)", "Only reviews written after delivery (QC)"],
    "on_time_avg": [ontime_scores.mean(), ontime_a.mean()],
    "late_avg": [late_scores.mean(), late_a.mean()],
    "gap_in_stars": [ontime_scores.mean() - late_scores.mean(), ontime_a.mean() - late_a.mean()],
    "effect_size_r": [rank_biserial(u_stat, len(ontime_scores), len(late_scores)), rank_biserial(u_a, len(ontime_a), len(late_a))],
    "p_value": [fmt_p(p_value), fmt_p(p_a)],
})
print("% of reviews written BEFORE the order arrived:")
print(before_share.round(1).to_string())
qc4.round(2)

In [ ]:
# QC6: state correlation without tiny states
big_states = by_state[by_state["orders"] >= 300]
r_all = stats.spearmanr(by_state["avg_delivery_days"], by_state["avg_review"])
r_big = stats.spearmanr(big_states["avg_delivery_days"], big_states["avg_review"])
print(f"All {len(by_state)} states: Spearman r = {r_all.correlation:.2f} ({fmt_p(r_all.pvalue)})")
print(f"{len(big_states)} states with 300+ orders: Spearman r = {r_big.correlation:.2f} ({fmt_p(r_big.pvalue)})")

# QC7: reconciliation checks
delivered_rev = fact_orders.loc[fact_orders["order_status"] == "delivered", "order_value"].sum()
recon = pd.DataFrame([
    ("RFM segments cover every customer", segment_summary["customers"].sum() == orders_per_customer.size),
    ("RFM revenue = total purchase value", np.isclose(segment_summary["revenue"].sum(), purchases["order_value"].sum())),
    ("Segment shares add to 100%", np.isclose(segment_summary["customers_%"].sum(), 100)),
    ("State revenue = delivered revenue", np.isclose(by_state["revenue"].sum(), delivered_rev, rtol=1e-4)),
    ("Category revenue = delivered item revenue", np.isclose(by_category["revenue"].sum(),
        fact_order_items.loc[fact_order_items["order_status"] == "delivered", "item_total"].sum(), rtol=1e-4)),
    ("Cohort sizes add up to all customers", cohort_size.sum() == orders_per_customer.size),
], columns=["check", "passed"])
recon["passed"] = recon["passed"].map({True: "✅ PASS", False: "❌ FAIL"})
recon

In [ ]:
# Revised findings after QC
late_row, ontime_row = qc2.loc["Late first order"], qc2.loc["On-time first order"]
qc_findings = [
    f"Repeat buying (counting purchase days): {(days_per_customer > 1).mean():.1%} of customers came back on a different day. "
    f"{same_day_only / len(repeat_customers):.0%} of the original 'repeat' customers had only split one shopping trip into several orders.",
    f"Among real repeat buyers, the median wait for a 2nd purchase is {gap_days_qc.median():.0f} days; "
    f"{(gap_days_qc <= 30).mean():.0%} return within 30 days.",
    f"Late delivery vs reviews holds with reviews written after delivery: {ontime_a.mean():.2f}★ vs {late_a.mean():.2f}★, "
    f"effect size r = {rank_biserial(u_a, len(ontime_a), len(late_a)):.2f} ({fmt_p(p_a)}).",
    f"Late first order vs coming back (same 180-day window): {late_row['returned_within_180d_pct']:.2f}% vs "
    f"{ontime_row['returned_within_180d_pct']:.2f}%; within cohort quarters the odds ratio is {or_mh:.2f} ({fmt_p(p_cmh)}), "
    f"so this finding is {'confirmed' if p_cmh < 0.05 and or_mh < 1 else 'NOT confirmed once timing is controlled for'}.",
    f"State delivery time vs reviews (states with 300+ orders): Spearman r = {r_big.correlation:.2f} ({fmt_p(r_big.pvalue)}).",
]
for i, f in enumerate(qc_findings, 1):
    print(f"QC{i}. {f}\n")

---
## Summary: what I did this week
1. **KPI layer:** one SQL query produces a monthly management table with written KPI definitions.
2. **Cohort retention:** tracked every monthly cohort for 12 months, plus fair repeat-rate windows (30/90/180/365 days).
3. **RFM segmentation:** scored every customer and translated the scores into 7 action-ready segments.
4. **Hypothesis testing:** used Mann-Whitney U and chi-square tests to check that late delivery really hurts reviews and repeat buying, rather than just eyeballing it.
5. **Revenue drivers:** broke revenue down by state and category with SQL window functions.
6. **Quality check:** stress-tested every finding (split orders, unequal observation windows, cohort timing, review timing, effect sizes, reconciliation) and revised the ones that didn't hold.

### Next: Week 3
An interactive **Streamlit dashboard** with a live link, built on these KPIs and segments.